# 01. Data Exploration

KSPHM-KIMM 2025 Bearing RUL Prediction - Data Exploration Notebook

This notebook covers:
1. Loading TDMS vibration data
2. Understanding data structure
3. Basic signal visualization
4. Frequency analysis overview

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timedelta

from src.preprocessing import TDMSDataLoader, load_tdms_segments
from src.analysis import FrequencyAnalyzer, BearingFaultFrequencies
from src.utils import plot_signal, plot_envelope_spectrum, plot_channel_comparison

# Print library versions
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")

## 1. Data Configuration

In [ ]:
# Configuration
DATA_DIR = "../data"
TRAIN_FOLDER = f"{DATA_DIR}/Train Set/Train1"

# Signal parameters
FS = 25600  # Sampling frequency (Hz)
CHANNELS = ["CH1", "CH2", "CH3", "CH4"]

# Bearing fault frequencies
FAULT_FREQS = BearingFaultFrequencies()
print("Fault Frequencies:")
for name, freq in FAULT_FREQS.as_dict().items():
    print(f"  {name}: {freq} Hz")

## 2. Load TDMS Data

In [ ]:
# Load all segments from a training folder
segments, timestamps = load_tdms_segments(TRAIN_FOLDER)

print(f"Number of segments: {len(segments)}")
print(f"Samples per segment: {len(segments[0])}")
print(f"Total duration: {timestamps[-1] - timestamps[0]}")

# Show first segment info
print(f"\nFirst segment columns: {segments[0].columns.tolist()}")
print(f"First timestamp: {timestamps[0]}")
print(f"Last timestamp: {timestamps[-1]}")

## 3. Concatenate Channel Data

In [ ]:
# Concatenate all segments for each channel
vib_data = {ch: np.concatenate([seg[ch].values for seg in segments]) for ch in CHANNELS}

total_samples = len(vib_data["CH1"])
total_duration = total_samples / FS

print(f"Total samples: {total_samples:,}")
print(f"Total duration: {total_duration:.2f} seconds ({total_duration/3600:.2f} hours)")

## 4. Visualize Raw Signals

In [ ]:
# Plot all channels
fig = plot_channel_comparison(
    vib_data,
    channels=CHANNELS,
    fs=FS,
    title="All Channels - First Segment",
    max_samples=256000
)
plt.show()

In [ ]:
# Plot single channel in detail
fig = plot_signal(
    vib_data["CH2"][:256000],
    fs=FS,
    title="CH2 - First 10 Seconds"
)
plt.show()

## 5. Basic Statistics

In [ ]:
# Calculate statistics for each channel
stats = []
for ch in CHANNELS:
    signal = vib_data[ch]
    stats.append({
        "Channel": ch,
        "Mean": np.mean(signal),
        "Std": np.std(signal),
        "Min": np.min(signal),
        "Max": np.max(signal),
        "RMS": np.sqrt(np.mean(signal**2))
    })

stats_df = pd.DataFrame(stats)
print("Channel Statistics:")
display(stats_df)

## 6. Frequency Domain Analysis

In [ ]:
from src.analysis import compute_envelope_stft

# Compute envelope spectrum for CH2
signal = vib_data["CH2"][:256000]
freqs, amplitudes = compute_envelope_stft(signal, FS, lowcut=1000, highcut=5000)

# Plot envelope spectrum
fig = plot_envelope_spectrum(
    freqs,
    amplitudes,
    fault_freqs=FAULT_FREQS,
    freq_limit=500,
    title="CH2 Envelope Spectrum (1000-5000Hz BPF)"
)
plt.show()

## 7. FFT Analysis

In [ ]:
# Direct FFT of raw signal
signal = vib_data["CH2"][:256000]

n = len(signal)
yf = np.fft.rfft(signal)
xf = np.fft.rfftfreq(n, 1/FS)
amplitude = np.abs(yf) / n * 2

# Plot FFT
plt.figure(figsize=(14, 5))
plt.plot(xf, amplitude, linewidth=0.5)
plt.xlabel("Frequency (Hz)")
plt.ylabel("Amplitude")
plt.title("CH2 - Raw FFT Spectrum")
plt.xlim(0, 5000)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Data Summary

In [ ]:
print("=" * 50)
print("DATA SUMMARY")
print("=" * 50)
print(f"Sampling Frequency: {FS} Hz")
print(f"Channels: {CHANNELS}")
print(f"Samples per segment: 256,000 (10 seconds)")
print(f"Total segments: {len(segments)}")
print(f"Total duration: {total_duration/3600:.2f} hours")
print("\nBearing Fault Frequencies:")
print(f"  BPFI (Inner race): {FAULT_FREQS.BPFI} Hz")
print(f"  BPFO (Outer race): {FAULT_FREQS.BPFO} Hz")
print(f"  BSF (Ball): {FAULT_FREQS.BSF} Hz")
print(f"  FTF (Cage): {FAULT_FREQS.FTF} Hz")
print("=" * 50)